# 3. Econometric Analysis: Stationarity Tests

This notebook conducts the preliminary statistical analysis required before model fitting. Its primary purpose is to verify that all time series are stationary, a critical assumption for VAR models.

The key steps are:
1.  **Load Transformed Data**: Executes the data processing notebook to load the `transformed_data` dictionary.
2.  **Define Reusable Test Function**: Creates a helper function to run the Augmented Dickey-Fuller (ADF) test and print the results in a standardized format.
3.  **Perform Stationarity Testing**: Applies the ADF test to each of the final transformed variables in the specified Cholesky order (`indpro`, `cpi`, `fedfunds`, `permits`, `unemp`, `hpi`).

This notebook confirms that our data is ready for the modeling phase.

## Interpreting results according to Gemini (why prefer pperron)




"For the SF and National series, both the ADF and the more robust PP test fail to reject the null of a unit root, providing strong evidence of non-stationary behavior in their growth rates. The only statistical fix for this would be to use the second difference. However, as this would make the results economically uninterpretable (price acceleration), this thesis will proceed with the first difference, acknowledging this persistent, bubble-like dynamic as a key feature of the data and a limitation of the linear VAR framework."

"While the ADF and PP tests produced conflicting results for some series, more weight is given to the PP test. The sample period contains a significant asset price bubble and regional structural breaks, conditions under which the variance of the series is unlikely to be constant. As the Phillips-Perron test is robust to such heteroskedasticity while the ADF test is not, the PP test is considered the more reliable indicator of stationarity for this analysis."

"For the Detroit-specific series, the Phillips-Perron test indicates stationarity while the ADF test does not. Given the clear evidence of a structural break in Detroit's economy around the year 2000, a condition known to cause heteroskedasticity, the PP test is considered the more appropriate and reliable of the two. Therefore, this analysis proceeds by treating the Detroit unemployment and permits series as stationary, based on the results from the more robust test." - Gemini




In [1]:
# Part 1: Load Transformed Data
# ==============================
# This cell executes the `2_data_processing.ipynb` notebook to load the
# `transformed_data` dictionary into the current environment.

%run ./2_data_processing.ipynb

# Import the ADF test function
from statsmodels.tsa.stattools import adfuller
# Import the Phillips-Perron test function
from arch.unitroot import PhillipsPerron

Loading cached data for National...
Loading cached data for SF...
Loading cached data for DET...
Loading cached data for ATL...
Loading cached data for LV...
\nData loading complete.
\nSample of raw data for SF:
             indpro    cpi  fedfunds        hpi  unemp      permits
date                                                               
1991-01-01  61.0842  134.7      6.91  71.935495    4.6  2037.040992
1991-02-01  60.6387  134.8      6.25  71.126390    4.7   585.889721
1991-03-01  60.2973  134.8      6.12  70.608208    4.9   835.829789
1991-04-01  60.4524  135.1      5.91  69.909268    5.0   789.687733
1991-05-01  61.0587  135.6      5.78  70.010559    5.1   625.659862
Processed 'indpro' (raw and log-difference).
Processed 'cpi' (raw and log-difference).
Processed 'fedfunds' (raw and first-difference).
Processed 'permits' for all areas (raw and log-difference).
Processed 'unemp' for all areas (raw and first-difference).
Processed 'hpi' for all areas (raw and log-difference).


In [2]:
# Part 2: Reusable ADF Test Function
# ===================================
# This function runs an Augmented Dickey-Fuller (ADF) test and prints the
# results in a clear, readable format.
#
# Null Hypothesis (H0): The series has a unit root (is non-stationary).
# We reject H0 if the p-value is less than 0.05.

def run_adf_test(series: pd.Series, series_name: str):
    """
    Performs an ADF test on a given series and prints a formatted result.

    Args:
        series (pd.Series): The time series to test.
        series_name (str): The name of the series for printing.
    """
    if series.empty:
        print(f"Cannot perform ADF test for {series_name}: Series is empty.")
        return

    print(f"--- ADF Test for {series_name} ---")
    result = adfuller(series.dropna())
    p_value = result[1]
    
    print(f"  Test Statistic: {result[0]:.4f}")
    print(f"  p-value: {p_value:.4f}")
    
    if p_value <= 0.05:
        print(f"  Conclusion: Stationary (Reject H0)")
    else:
        print(f"  Conclusion: Non-Stationary (Fail to Reject H0)")
    print("-" * 40)

In [3]:
# Part 2b: Reusable Phillips-Perron Test Function
# ==============================================
# This function runs a Phillips-Perron (PP) test, which is another type of
# unit root test.
#
# Null Hypothesis (H0): The series has a unit root (is non-stationary).
# We reject H0 if the p-value is less than 0.05.

def run_pp_test(series: pd.Series, series_name: str):
    """
    Performs a Phillips-Perron test on a given series and prints a formatted result.

    Args:
        series (pd.Series): The time series to test.
        series_name (str): The name of the series for printing.
    """
    if series.empty:
        print(f"Cannot perform PP test for {series_name}: Series is empty.")
        return

    print(f"--- Phillips-Perron Test for {series_name} ---")
    result = PhillipsPerron(series.dropna())
    p_value = result.pvalue
    
    print(f"  Test Statistic: {result.stat:.4f}")
    print(f"  p-value: {p_value:.4f}")
    
    if p_value <= 0.05:
        print(f"  Conclusion: Stationary (Reject H0)")
    else:
        print(f"  Conclusion: Non-Stationary (Fail to Reject H0)")
    print("-" * 40)

# Part 3: Perform Stationarity Tests
 We now apply the ADF test to our final transformed variables to confirm
they are all stationary. The tests are performed in the Cholesky order.

### 3.1 Industrial Production (indpro_log_diff)

In [4]:
if 'indpro_log_diff' in transformed_data:
    run_adf_test(transformed_data['indpro_log_diff'], 'Log-Differenced Industrial Production')
else:
    print("Transformed 'indpro' data not found.")

--- ADF Test for Log-Differenced Industrial Production ---
  Test Statistic: -6.2911
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------


In [5]:
if 'indpro_log_diff' in transformed_data:
    run_pp_test(transformed_data['indpro_log_diff'], 'Log-Differenced Industrial Production')
else:
    print("Transformed 'indpro' data not found.")

--- Phillips-Perron Test for Log-Differenced Industrial Production ---
  Test Statistic: -15.2213
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------


### 3.2 Consumer Price Index (cpi_log_diff)

In [6]:
if 'cpi_log_diff' in transformed_data:
    run_adf_test(transformed_data['cpi_log_diff'], 'Log-Differenced CPI')
else:
    print("Transformed 'cpi' data not found.")

--- ADF Test for Log-Differenced CPI ---
  Test Statistic: -3.4348
  p-value: 0.0098
  Conclusion: Stationary (Reject H0)
----------------------------------------


In [7]:
if 'cpi_log_diff' in transformed_data:
    run_pp_test(transformed_data['cpi_log_diff'], 'Log-Differenced CPI')
else:
    print("Transformed 'cpi' data not found.")

--- Phillips-Perron Test for Log-Differenced CPI ---
  Test Statistic: -11.1446
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------


### 3.3 Federal Funds Rate (fedfunds_diff)

In [8]:
if 'fedfunds_diff' in transformed_data:
    run_adf_test(transformed_data['fedfunds_diff'], 'First-Differenced Federal Funds Rate')
else:
    print("Transformed 'fedfunds' data not found.")

--- ADF Test for First-Differenced Federal Funds Rate ---
  Test Statistic: -3.7530
  p-value: 0.0034
  Conclusion: Stationary (Reject H0)
----------------------------------------


In [9]:
if 'fedfunds_diff' in transformed_data:
    run_pp_test(transformed_data['fedfunds_diff'], 'First-Differenced Federal Funds Rate')
else:
    print("Transformed 'fedfunds' data not found.")

--- Phillips-Perron Test for First-Differenced Federal Funds Rate ---
  Test Statistic: -9.2420
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------


### 3.4 Building Permits (permits_log_diff)

In [10]:
if 'permits_log_diff' in transformed_data:
    for area in transformed_data['permits_log_diff'].columns:
        series = transformed_data['permits_log_diff'][area]
        run_adf_test(series, f'Log-Differenced Permits in {area}')
else:
    print("Transformed 'permits' data not found.")

--- ADF Test for Log-Differenced Permits in National ---
  Test Statistic: -15.8928
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced Permits in SF ---
  Test Statistic: -7.6293
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced Permits in DET ---
  Test Statistic: -1.2057
  p-value: 0.6712
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced Permits in ATL ---
  Test Statistic: -16.9062
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced Permits in LV ---
  Test Statistic: -4.8932
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------


In [11]:
if 'permits_log_diff' in transformed_data:
    for area in transformed_data['permits_log_diff'].columns:
        series = transformed_data['permits_log_diff'][area]
        run_pp_test(series, f'Log-Differenced Permits in {area}')
else:
    print("Transformed 'permits' data not found.")

--- Phillips-Perron Test for Log-Differenced Permits in National ---
  Test Statistic: -15.9467
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced Permits in SF ---
  Test Statistic: -51.8672
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced Permits in DET ---
  Test Statistic: -22.6343
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced Permits in ATL ---
  Test Statistic: -28.9494
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced Permits in LV ---
  Test Statistic: -26.3774
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------


### 3.5 Unemployment (unemp_diff)

In [12]:
if 'unemp_diff' in transformed_data:
    for area in transformed_data['unemp_diff'].columns:
        series = transformed_data['unemp_diff'][area]
        run_adf_test(series, f'First-Differenced Unemployment in {area}')
else:
    print("Transformed 'unemp' data not found.")

--- ADF Test for First-Differenced Unemployment in National ---
  Test Statistic: -3.6765
  p-value: 0.0045
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- ADF Test for First-Differenced Unemployment in SF ---
  Test Statistic: -3.4166
  p-value: 0.0104
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- ADF Test for First-Differenced Unemployment in DET ---
  Test Statistic: -2.0743
  p-value: 0.2550
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- ADF Test for First-Differenced Unemployment in ATL ---
  Test Statistic: -8.0774
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- ADF Test for First-Differenced Unemployment in LV ---
  Test Statistic: -4.1845
  p-value: 0.0007
  Conclusion: Stationary (Reject H0)
----------------------------------------


In [13]:
if 'unemp_diff' in transformed_data:
    for area in transformed_data['unemp_diff'].columns:
        series = transformed_data['unemp_diff'][area]
        run_pp_test(series, f'First-Differenced Unemployment in {area}')
else:
    print("Transformed 'unemp' data not found.")

--- Phillips-Perron Test for First-Differenced Unemployment in National ---
  Test Statistic: -16.3440
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for First-Differenced Unemployment in SF ---
  Test Statistic: -9.4463
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for First-Differenced Unemployment in DET ---
  Test Statistic: -7.5478
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for First-Differenced Unemployment in ATL ---
  Test Statistic: -12.0897
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for First-Differenced Unemployment in LV ---
  Test Statistic: -9.1769
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------


### 3.6 House Price Index (hpi_log_diff)

In [14]:
if 'hpi_log_diff' in transformed_data:
    for area in transformed_data['hpi_log_diff'].columns:
        series = transformed_data['hpi_log_diff'][area]
        run_adf_test(series, f'Log-Differenced HPI in {area}')
else:
    print("Transformed 'hpi' data not found.")

--- ADF Test for Log-Differenced HPI in National ---
  Test Statistic: -1.5225
  p-value: 0.5223
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced HPI in SF ---
  Test Statistic: -2.5397
  p-value: 0.1061
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced HPI in DET ---
  Test Statistic: 3.5639
  p-value: 1.0000
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced HPI in ATL ---
  Test Statistic: 0.1336
  p-value: 0.9683
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- ADF Test for Log-Differenced HPI in LV ---
  Test Statistic: -3.6055
  p-value: 0.0057
  Conclusion: Stationary (Reject H0)
----------------------------------------


In [15]:
if 'hpi_log_diff' in transformed_data:
    for area in transformed_data['hpi_log_diff'].columns:
        series = transformed_data['hpi_log_diff'][area]
        run_pp_test(series, f'Log-Differenced HPI in {area}')
else:
    print("Transformed 'hpi' data not found.")

--- Phillips-Perron Test for Log-Differenced HPI in National ---
  Test Statistic: -2.1030
  p-value: 0.2433
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced HPI in SF ---
  Test Statistic: -2.6111
  p-value: 0.0907
  Conclusion: Non-Stationary (Fail to Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced HPI in DET ---
  Test Statistic: -6.6254
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced HPI in ATL ---
  Test Statistic: -8.9128
  p-value: 0.0000
  Conclusion: Stationary (Reject H0)
----------------------------------------
--- Phillips-Perron Test for Log-Differenced HPI in LV ---
  Test Statistic: -3.7734
  p-value: 0.0032
  Conclusion: Stationary (Reject H0)
----------------------------------------
